# 06 · BraTS 2021 data check

Data for the 3D segmentation model: [BraTS 2021 Task 1](https://www.kaggle.com/datasets/dschettler8845/brats-2021-task1), 4 MRI types (FLAIR, T1, T1ce, T2) + an expert tumor mask per patient.

1. Clone the repo and install the package (+ MONAI).
2. Run `python -m neurolens_ml.prep_brats`: unpack the `.tar` files, find every case, read headers and masks, load one case with MONAI.
3. Show the report: case count, shapes, label values, tumor volumes.

**Kaggle setup:** no GPU needed, Internet on, input `dschettler8845/brats-2021-task1`. Run with *Save Version → Save & Run All*. Download `brats_report.json` and `brats_cases.csv` from *Output*.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sanjardevx/neurolens.git"
BRANCH = "develop"
REPO_DIR = Path(os.environ.get("NEUROLENS_REPO_DIR", "/tmp/neurolens"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
EXTRA_ARGS = os.environ.get("NEUROLENS_ARGS", "").split()  # only for local smoke tests (fake data paths)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
# --no-deps: keep Kaggle's own torch and numpy; MONAI and nibabel need nothing else
missing = [p for p in ["neurolens_ml", "monai", "nibabel"] if importlib.util.find_spec(p) is None]
if "neurolens_ml" in missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR / "ml")], check=True)
if pkgs := [p for p in missing if p != "neurolens_ml"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pkgs], check=True)
print("repo commit:", commit)

In [ ]:
cmd = [sys.executable, "-m", "neurolens_ml.prep_brats", "--out", str(OUT_DIR), *EXTRA_ARGS]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "data check failed, see the log above"

In [ ]:
import pandas as pd

report = json.loads((OUT_DIR / "brats_report.json").read_text())
report["repo_commit"] = commit
(OUT_DIR / "brats_report.json").write_text(json.dumps(report, indent=2))
print("cases:", report["n_cases"], "| labels:", report["label_values"], "| unknown:", report["unknown_labels"])
print("no tumor:", report["cases_without_tumor"], "| no enhancing part:", report["cases_without_enhancing"])
cases = pd.read_csv(OUT_DIR / "brats_cases.csv")
cases[["wt_cm3", "tc_cm3", "et_cm3"]].describe().round(1)